# Synthesis

What does the whole study say? Group A describes what decision models are like (outputs, stability, geometry); Group B asks what their probabilities add to tabular predictors compared with frozen embeddings, label-trained alternatives and native-text systems; Group C looks inside Strands and trains it on labels. The synthesis brings together one study-at-a-glance figure, the 21 prespecified comparisons, and the status of the planned paper figures. Datasets are the unit (18: 9 classification, 9 regression); raw gains stay within a task family and only the headroom-normalised gain and the MulTaBench-style normalised score pool all 18.

In [1]:
import sys
from pathlib import Path
REPO_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").is_file() and (p / "src").is_dir())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
import matplotlib.pyplot as plt
import pandas as pd
from src.utils.notebook_display import display_local, display_json, display_table
from src.utils.notebook_report import begin_report, report_section, record, finish_report
from src.utils.provenance import provenance
from src.visualize import figures, style
style.apply()

from src.evaluation import learner_scores, study_audit
from src.evaluation.synthesis_overview import glance_table
from src.visualize.overview import confirmatory_forest, study_at_a_glance
NOTEBOOK = "general/4_synthesis"
begin_report()
save = figures.FigureSaver(NOTEBOOK)
report_section("1. Scope and provenance")
display_json("Notebook execution identity", provenance())
exports = study_audit.export_status()
frames = exports.attrs["frames"]
display_table("Result exports read by this synthesis", exports[["study", "notebook", "status", "created_utc"]], preview_rows=12)

study,notebook,status,created_utc
A1,A/A1,superseded_schema,2026-11-18T13:24:39.664228+00:00
A2,A/A2,awaiting_export,None
A3,A/A3,awaiting_export,None
B1,B/B1,awaiting_export,None
B2,B/B2,awaiting_export,None
B3,B/B3,awaiting_export,None
B4,B/B4,awaiting_export,None
B5,B/B5,awaiting_export,None
B6,B/B6,awaiting_export,None
C1,C/C1,awaiting_export,None


## Study at a glance

For each prespecified arm, the gain over the same predictor without text: mean and median with dataset-bootstrap intervals; pooled headroom-normalised gain, the pooled MulTaBench-style normalised score (once the fold tables exist: per dataset and fold, min-max over every predictor × arm shown plus no text, negative R² clipped at 0, averaged over predictors; relative to that set) and raw ΔAUC / ΔR² per task family in separate panels. Decision rows use the median over every member of the declared decision-model cohort; the Strands row is the prespecified LoRA + head cell (no winner is selected).

In [2]:
report_section("2. Study at a glance")
glance = glance_table({s: frames.get(s) or {} for s in ("B1", "B2", "B3", "C2")})
if glance.empty:
    print("Waiting for export: the B1, B2, B3 and C2 exports are not held locally.")
else:
    display_table("Study at a glance", glance.round(4), preview_rows=20)
    fig, name, caption = study_at_a_glance(glance)
    save(fig, name, caption=caption)
    display_local(fig)
    plt.close(fig)

Waiting for export: the B1, B2, B3 and C2 exports are not held locally.


## The 21 primary comparisons

For each of seven predictors, compare decision features with three alternatives: the structured baseline, frozen E5-small embeddings, and label-trained E5-small TAR embeddings. The decision-feature result uses the median across models in the release. All comparisons use the same 18 datasets and five folds. The two-sided Wilcoxon tests share one Holm correction across all 21 comparisons. The analysis reports missing inputs until their exports are complete.

In [3]:
report_section("3. Statistical framework")
state = learner_scores.framework_state()
family = learner_scores.confirmatory_family_state()
print(f"framework.status = {state['framework_status']}; approved: {state['approved']}")
print(f"Confirmatory family: {family['status']}; {family['declared_contrasts']} declared contrasts "
      f"({family['pooled_contrasts']} pooled, {family['per_task_contrasts']} per task).")
display_table("Planned learner comparisons per experiment", learner_scores.applications_table(), preview_rows=12)
display_json("Confirmatory family state", family)
confirmatory = learner_scores.run_confirmatory_family()
print(f"Confirmatory family run: {confirmatory['status']}"
      + (f"; waiting for {', '.join(confirmatory['missing'])}" if confirmatory.get("missing") else "")
      + (f"; {confirmatory['reason']}" if confirmatory.get("reason") else ""))
display_json("Decision-model cohort for all primary comparisons", {
    key: confirmatory[key] for key in ("decision_cohort", "decision_roster", "decision_models")
})
if confirmatory["status"] == "compared":
    print(learner_scores.publish_confirmatory(confirmatory))
    display_table("Confirmatory family: two-sided Wilcoxon, Holm over every declared contrast, Bayesian signed-rank", confirmatory["results"], preview_rows=21)
    fig, name, caption = confirmatory_forest(confirmatory["forest"])
    save(fig, name, caption=caption)
    display_local(fig)
    plt.close(fig)

framework.status = approved; approved: True
Confirmatory family: ready_to_run; 21 declared contrasts (21 pooled, 0 per task).


application,comparison,procedure,inference,holm_family,score,export,gate
A1_outputs,selected_decision_model_cohort,friedman_ranks_cd_diagram,descriptive,None,log_loss,A1,approved
A2_stability,averaged_presentations_vs_canonical_per_model,wilcoxon_holm_within_factor_over_selected_models_plus_bayesian,confirmatory,selected_models_vs_original_within_each_factor,proper_score_skill,A2,approved
A3_geometry,None,None,descriptive_only,None,None,None,approved
B1_decision_features,decision_modes_vs_non_text_control_per_predictor,wilcoxon_holm_plus_bayesian,confirmatory,decision_modes_vs_non_text_within_each_predictor,headroom_normalised_gain,B1,approved
B2_frozen_embeddings,decision_vs_frozen_embeddings_and_decision_plus_frozen,wilcoxon_holm_plus_bayesian,confirmatory,frozen_representations_vs_decision_control_within_each_predictor,headroom_normalised_gain,B2,approved
B3_label_trained,decision_vs_tar_and_label_trained_classifier,wilcoxon_holm_plus_bayesian,confirmatory,label_trained_representations_vs_decision_control_within_each_predictor,headroom_normalised_gain,B3,approved
B4_when_text_helps,three_lodo_rules_vs_always_and_never,wilcoxon_on_regret_plus_bayesian,exploratory,None,regret,B4,approved
B5_data_efficiency,arms_vs_baseline_per_size,wilcoxon_holm_vs_control,exploratory,None,headroom_normalised_gain,B5,approved
B6_native_text_systems,native_text_systems,friedman_cd_diagram,descriptive,None,headline_metric,B6,approved
C1_strands_states,14_arms_vs_non_text_control,friedman_then_wilcoxon_holm_vs_control,exploratory,None,headroom_normalised_gain,C1,approved


Confirmatory family run: awaiting_export; waiting for B1_decision_features, B2_frozen_embeddings, B3_label_trained


## Paper figures

The index links the proposed evidence to its owning notebook. Start with the three paired utility comparisons per predictor and their dataset-level variation. The representation ladder and moderator analyses add context. Qualification graphics describe the pipeline. A missing result stays pending until its export arrives. Final manuscript placement remains the authors’ decision.

In [4]:
report_section("4. Paper figures")
from src.utils.paper_figures import resolve
display_table("Paper figure registry", resolve(), preview_rows=20)

id,title,notebook,kind,placement,planned_figures,saved_figures,status,note
fig1_headline_gain,What decision models add - per-dataset gains and model ranks,B/B1,figure,undecided,"b1_gain_landscape, b1_model_ranks",pending,planned,The per-dataset effect is the headline view. Ranks add context about heterogeneity across extractors.
fig2_representation_ladder_and_compute,"Decision features against embedding, label-trained and native alternatives, and compute",B/B_synthesis,figure,undecided,"b_ladder_*, b_quality_vs_compute",pending,planned,Select the ladder that answers the comparison in the text. Extraction time has its own request denominator.
fig3_dataset_and_model_explanations,When do decision features help - dataset moderators and decision models,B/B4,figure,undecided,"b4_headroom_and_properties, b4_model_diagnostics_vs_usefulness",pending,planned,Exploratory associations use datasets or models as indicated. Their sampling units differ.
fig4_fields_and_prediction_in_advance,"Which text columns help, and predicting usefulness in advance",B/B4,figure,undecided,"b4_text_column_refits, b4_lodo_rule",pending,planned,Refitting after removing one text column measures its contribution. The rule is evaluated on held-out datasets.
fig_primary_contrasts,"Paired utility against no text, frozen E5-small and supervised TAR",general/4_synthesis,figure,undecided,confirmatory_family,pending,available_when_results_exist,"All 21 prespecified comparisons, with the dataset-level effect and uncertainty for the recorded cohort."
fig_study_at_a_glance,Study at a glance - every prespecified arm against no text,general/4_synthesis,figure,undecided,study_at_a_glance,pending,available_when_results_exist,
c1_representations_panel,"Inside the decision model (Strands hidden states, Group C1)",C/C1,figure,undecided,"c1_position_path, c1_contrasts, c1_vector_vs_probabilities_by_dataset, c1_gains_per_column, c1_gains_joint",pending,available_when_results_exist,
tab_rq3_adaptation,Learning from labels with Strands (Group C2),C/C2,figure,undecided,"c2_factorial, c2_cells, c2_context, c2_datasets, c2_gains",pending,available_when_results_exist,
fig_strands_ladder,Strands from zero-shot probabilities to label training,C/C_synthesis,figure,undecided,strands_ladder,pending,available_when_results_exist,


## Complete report

In [5]:
finish_report(NOTEBOOK, save.summary() + "\nOnly verified exports and configuration were read; no inference, model training or live VSC changes were performed.")

general/4_synthesis

1. Scope and provenance


Notebook execution identity
{
  "code_sha256": "253d5f17bdea4be35b0e853055326f9a06bd5d0a5b2b0c789f83d236fd3c5074",
  "git_head": "0929454751c9921410a46b29506eb6c2b4ca280f",
  "git_dirty": true,
  "tfm_library_pin": "4574eea0c2ce6646b5d2e9c7f53a6ef3882c368c dm-library (remotes/origin/HEAD)\n 248f137046487e0d57360abba1e060420ae54e1a tfm-library (remotes/origin/HEAD)\n 2208cf68d6351eff2cb0ddb5a9eb1d822523457b vsc-library (heads/main-8-g2208cf6)",
  "environment": {
    "python": "3.12.10",
    "numpy": "2.5.4",
    "pandas": "2.3.3",
    "PyYAML": "6.0.3",
    "matplotlib": "3.11.2",
    "pyarrow": "25.0.1"
  }
}


Result exports read by this synthesis
study notebook            status                      created_utc
   A1     A/A1 superseded_schema 2026-11-18T13:24:39.664228+00:00
   A2     A/A2   awaiting_export                             None
   A3     A/A3   awaiting_export                             None
   B1     B/B1   awaiting_expor